# Urdu TTS — Piper fine-tune (Colab GPU via VS Code)

Fine-tunes a Piper voice on 4,129 Common Voice Urdu clips, starting from the Arabic `kareem-medium` checkpoint.

Runs against a Colab GPU runtime. Files move in/out through the `/content` folder, which you can browse in the VS Code Explorer.

Run cells top to bottom. Training (cell 6) runs until you stop it; then export with cell 7 and download the result from `/content`.


## 1. Check GPU


In [19]:
!nvidia-smi -L


GPU 0: Tesla T4 (UUID: GPU-986dc5b7-1d5e-e41b-bf10-b741f78eb40a)


## 2. Install Piper training code

Uses the fork's own pinned versions (PyTorch Lightning 2.4, torch 2.4+cu121), with two fixes for modern Colab Python:
`piper-phonemize-fix` (provides the `piper_phonemize` wheel for Python 3.11/3.12) and `numpy 1.26`.


In [20]:
!git clone -q https://github.com/rmcpantoja/piper
%cd /content/piper/src/python

!pip install -q piper-phonemize-fix
!grep -viE 'piper-phonemize|^numpy' requirements.txt > /content/reqs.txt
!pip install -q -r /content/reqs.txt numpy==1.26.4
!bash build_monotonic_align.sh

!python -c "from piper_phonemize import phonemize_espeak as p; import torch, pytorch_lightning as pl; \
print('phonemize OK | torch', torch.__version__, '| pl', pl.__version__, '| cuda', torch.cuda.is_available()); \
print('ur test:', ''.join(sum(p('سلام دنیا','ur'), [])))"


fatal: destination path 'piper' already exists and is not an empty directory.
/content/piper/src/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 40.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 86.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 

## 3. Get the dataset onto the runtime

The Colab GPU is a remote machine — it can't see files on your computer, so we send the **single zip** to it (far more reliable than pasting 4,000 loose files).

1. Upload `urdu_dataset.zip` to Google Drive.
2. Right-click it → Share → **Anyone with the link** → Copy link.
3. Paste the link into `DRIVE_LINK` below and run this cell.


In [21]:
import os, glob

DRIVE_LINK = 'https://drive.google.com/file/d/1IzXtIyH1_wtufhJ3s-vyQ7VrX6FGgUvj/view?usp=drive_link'  # e.g. 'https://drive.google.com/file/d/1AbCdEf.../view?usp=sharing'

if DRIVE_LINK and not os.path.exists('/content/urdu_dataset.zip'):
    os.system('pip install -q gdown')
    os.system(f"gdown '{DRIVE_LINK}' --fuzzy -O /content/urdu_dataset.zip")

assert os.path.exists('/content/urdu_dataset.zip'), \
    'urdu_dataset.zip is not on the runtime. Set DRIVE_LINK above (or drag the single zip into /content), then re-run.'

os.system('unzip -q -o /content/urdu_dataset.zip -d /content/urdu_dataset')
DATASET_DIR = '/content/urdu_dataset'
nwav = len(glob.glob(DATASET_DIR + '/wavs/*.wav'))
print('DATASET_DIR =', DATASET_DIR, '| wavs:', nwav)
assert nwav > 0, 'Unzip produced no wavs — re-check the uploaded zip.'


DATASET_DIR = /content/urdu_dataset | wavs: 4129


## 4. Download the base checkpoint (Arabic medium)

What we fine-tune from — Arabic shares Urdu's script and much of its phonology.


In [22]:
!wget -q --show-progress \
  'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ar/ar_JO/kareem/medium/epoch=5079-step=1682020.ckpt' \
  -O /content/pretrained.ckpt
!ls -lh /content/pretrained.ckpt


/content/pretrained 100%[===================>] 806.71M  56.8MB/s    in 13s     
-rw-r--r-- 1 root root 807M Jun 25 05:16 /content/pretrained.ckpt


## 5. Preprocess with the Urdu phonemizer (`ur`)

This is the fix vs. the old run, which used `ne` (Nepali) and produced garbage. `--sample-rate 22050` upsamples the 16 kHz clips to match the medium base.


In [23]:
%cd /content/piper/src/python
!python -m piper_train.preprocess \
  --language ur \
  --input-dir '{DATASET_DIR}' \
  --output-dir /content/output \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050 \
  --max-workers 2

import json
cfg = json.load(open('/content/output/config.json'))
print('espeak voice:', cfg['espeak']['voice'], '| sample_rate:', cfg['audio']['sample_rate'], '| num_symbols:', cfg['num_symbols'])


/content/piper/src/python
INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 4129 utterance(s) with 2 worker(s)
min value is  tensor(-1.0103)
max value is  tensor(1.0202)
min value is  tensor(-1.0028)
max value is  tensor(1.0087)
min value is  tensor(-1.0401)
max value is  tensor(1.0040)
min value is  tensor(-1.0037)
max value is  tensor(1.0018)
min value is  tensor(-1.0546)
max value is  tensor(1.0401)
max value is  tensor(1.0050)
max value is  tensor(1.0011)
max value is  tensor(1.0024)
max value is  tensor(1.0004)
min value is  tensor(-1.0341)
max value is  tensor(1.0181)
min value is  tensor(-1.0033)
max value is  tensor(1.0166)
min value is  tensor(-1.0042)
espeak voice: ur | sample_rate: 22050 | num_symbols: 256


After this, `espeak voice` must print **`ur`**. If it does, the phonemes are correct.


## 6. Train

Saves a checkpoint every epoch plus `last.ckpt`. Let it run for a few hours, then stop it (the latest checkpoint is what you export). If you hit out-of-memory, lower `--batch-size` to 12 or 8.


In [ ]:
!python -m piper_train \
  --dataset-dir /content/output \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --quality medium \
  --validation-split 0.01 --num-test-examples 2 \
  --checkpoint-epochs 1 --num_ckpt 1 --save_last True \
  --log_every_n_steps 250 \
  --max_epochs 10000 \
  --resume_from_checkpoint /content/pretrained.ckpt \
  --precision 32


DEBUG:piper_train:Namespace(dataset_dir='/content/output', checkpoint_epochs=1, quality='medium', resume_from_single_speaker_checkpoint=None, batch_size=16, validation_split=0.01, num_test_examples=2, max_phoneme_ids=None, hidden_channels=192, inter_channels=192, filter_channels=768, n_layers=6, n_heads=2, lr_decay=0.999875, lr_reduce_enabled=False, lr_reduce_factor=0.5, lr_reduce_patience=10, show_plot=False, plot_save_path=None, learning_rate=0.0002, weight_decay=0.01, override_learning_rate=False, grad_clip=None, accelerator='gpu', devices=1, log_every_n_steps=250, max_epochs=10000, seed=1234, random_seed=False, resume_from_checkpoint='/content/pretrained.ckpt', precision='32', num_ckpt=1, default_root_dir=None, save_last=True, monitor='val_loss', monitor_mode='min', early_stop_patience=0)
DEBUG:piper_train:Using manual seed: 1234
DEBUG:piper_train:Checkpoints will be saved every 1 epoch(s)
DEBUG:piper_train:1 Checkpoints will be saved
GPU available: True (cuda), used: True
TPU avai

## 7. Export to ONNX

Stop cell 6 first. This writes `/content/urdu.onnx` and `/content/urdu.onnx.json` — **download both from the `/content` folder in the VS Code Explorer** (right-click → Download).


In [ ]:
import glob, os
ckpts = glob.glob('/content/output/lightning_logs/**/checkpoints/*.ckpt', recursive=True)
last = [c for c in ckpts if 'last' in c]
ckpt = (last or sorted(ckpts, key=os.path.getmtime))[-1]
print('exporting:', ckpt)

!python -m piper_train.export_onnx '{ckpt}' /content/urdu.onnx
!cp /content/output/config.json /content/urdu.onnx.json
!ls -lh /content/urdu.onnx /content/urdu.onnx.json
print('Done. Download both files from /content in the Explorer.')


---
Bring `urdu.onnx` + `urdu.onnx.json` back to your machine and synthesize:
```bash
echo 'یہ اردو میں بولنے والی آواز ہے' | piper --model urdu.onnx --output_file test.wav
```
